# Clinical RAG Pipeline — Improved

Same PDF → TOC-aware sections → chunk → dedup → hybrid retrieval pipeline as
`Version12PM.ipynb`, with the following fixes applied:

1. **MMR → similarity search** for the vector retriever (MMR was trading away
   top-relevance results for diversity, hurting precision)
2. **`EnsembleRetriever` removed** (it was built but never actually used — the
   fusion below replaces it with an explicit, tunable version)
3. **Real similarity scores** instead of rank position for confidence scoring
4. **Cross-encoder reranking** stage added on top of the fused candidate pool
5. **Larger chunks** (900 chars / 150 overlap, up from 500/100) so enumerated
   answers don't get split across chunk boundaries
6. **Tuned BM25 params** (`k1=1.8, b=0.85`) for dense clinical prose
7. **Abbreviation expansion for the BM25 leg only** (embeddings already
   understand "IE" ≈ "infective endocarditis"; BM25 doesn't)
8. **Source filtering** for guideline-specific questions ("...the 2023 ESC
   guidelines...")
9. **Non-truncated, hash-based chunk IDs** (the old `section[:20]` truncation
   risked collisions between similarly-named sections)

The last cell runs your original config and this improved config back-to-back
on the same eval set so you can see the actual delta.


## 0. Install & imports

In [1]:
!pip install -q pymupdf langchain-text-splitters langchain-community langchain-classic langchain-huggingface chromadb rank_bm25 sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 81.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 105.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 83.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 113.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.7/94

In [2]:
import pymupdf
import re
import hashlib
from difflib import SequenceMatcher

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_huggingface import HuggingFaceEmbeddings
from sentence_transformers import CrossEncoder


/tmp/ipykernel_746/1323470819.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


## 1. PDF → TOC-aware sections (unchanged from original)

In [3]:
EXCLUDE_SECTION_KEYWORDS = [
    "appendix", "references", "bibliography", "acknowledgements",
    "table of contents", "list of tables", "list of figures",
    "author information", "supplementary data"
]

def extract_and_merge_pdf(pdf_path):
    doc = pymupdf.open(pdf_path)
    toc = doc.get_toc()

    page_to_section = {}
    for item in toc:
        level, title, page = item
        if page not in page_to_section:
            page_to_section[page] = title

    merged_sections = {}
    current_section = "Front Matter"

    for page_num in range(len(doc)):
        actual_page = page_num + 1
        if actual_page in page_to_section:
            current_section = page_to_section[actual_page]

        page_text = doc.load_page(page_num).get_text("text")

        if current_section not in merged_sections:
            merged_sections[current_section] = {
                "source": pdf_path.split("/")[-1],
                "page": actual_page,
                "section": current_section,
                "text": ""
            }
        merged_sections[current_section]["text"] += "\n" + page_text

    final_data = []
    excluded_log = []
    for section_name, data in merged_sections.items():
        section_lower = section_name.lower()
        if any(kw in section_lower for kw in EXCLUDE_SECTION_KEYWORDS):
            excluded_log.append(section_name)
            continue

        text = data["text"]

        text = re.sub(r'https?://\S+', '', text)
        text = re.sub(r'www\.\S+', '', text)
        text = re.sub(r'doi:\s*\S+', '', text, flags=re.IGNORECASE)

        text = re.sub(r'(\w+)-\n(\w+)', r'\1\2', text)
        text = re.sub(r'(?<!\n)\n(?!\n)', ' ', text)
        text = re.sub(r'\s{2,}', ' ', text)
        cleaned_text = text.strip()

        if cleaned_text and len(cleaned_text) > 50:
            data["text"] = cleaned_text
            final_data.append(data)

    print(f"[{pdf_path.split('/')[-1]}] Kept {len(final_data)} sections, excluded {len(excluded_log)}: {excluded_log}")
    return final_data


In [4]:
pdf_files = ["ESC.pdf", "NICE.pdf"]
all_documents = []

for pdf_path in pdf_files:
    file_sections = extract_and_merge_pdf(pdf_path)
    all_documents.extend(file_sections)

print(f"\nTotal merged, cleaned, non-reference sections: {len(all_documents)}")


[ESC.pdf] Kept 50 sections, excluded 2: ['18. Supplementary data', 'Appendix']
[NICE.pdf] Kept 38 sections, excluded 0: []

Total merged, cleaned, non-reference sections: 88


## 2. Chunking — FIX #5, #9: bigger chunks, non-truncated IDs

`chunk_size` 500→900, `chunk_overlap` 100→150 so multi-sentence enumerated
answers (e.g. "four cardiac conditions...") are less likely to be split
across chunk boundaries. `chunk_id` now uses a hash of the full section name
instead of `section[:20]`, so two sections with similar first-20-characters
can't collide.


In [5]:
def chunk_merged_sections(all_documents, chunk_size=800, chunk_overlap=150):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""]
    )

    final_chunks = []
    for doc in all_documents:
        section_chunks = text_splitter.split_text(doc["text"])
        section_hash = hashlib.md5(doc["section"].encode()).hexdigest()[:8]
        for i, chunk_text in enumerate(section_chunks):
            chunk_data = {
                "source": doc["source"],
                "section": doc["section"],
                "section_start_page": doc["page"],
                "chunk_id": f"{doc['source']}_Sec_{section_hash}_Chunk_{i+1}",
                "text": chunk_text,
                "text_hash": hashlib.md5(chunk_text.strip().lower().encode()).hexdigest()
            }
            final_chunks.append(chunk_data)

    return final_chunks

rag_chunks = chunk_merged_sections(all_documents)
print(f"Total raw chunks before dedup: {len(rag_chunks)}")


Total raw chunks before dedup: 947


## 3. Dedup (unchanged from original)

In [6]:
def deduplicate_chunks(chunks, near_dup_threshold=0.92):
    seen_hashes = set()
    exact_deduped = []
    for c in chunks:
        if c["text_hash"] not in seen_hashes:
            seen_hashes.add(c["text_hash"])
            exact_deduped.append(c)

    print(f"Removed {len(chunks) - len(exact_deduped)} exact duplicates")

    final = []
    kept_texts_by_source = {}
    near_dup_removed = 0

    for c in exact_deduped:
        src = c["source"]
        kept_texts_by_source.setdefault(src, [])
        is_dup = False
        for prev_text in kept_texts_by_source[src]:
            ratio = SequenceMatcher(None, c["text"][:300], prev_text[:300]).ratio()
            if ratio >= near_dup_threshold:
                is_dup = True
                near_dup_removed += 1
                break
        if not is_dup:
            kept_texts_by_source[src].append(c["text"])
            final.append(c)

    print(f"Removed {near_dup_removed} near-duplicates")
    print(f"Final deduplicated chunk count: {len(final)}")
    return final

rag_chunks = deduplicate_chunks(rag_chunks)


Removed 0 exact duplicates
Removed 15 near-duplicates
Final deduplicated chunk count: 932


## 4. Indexing — FIX #1, #6: similarity search (not MMR), tuned BM25

- `vector_retriever` now uses `search_type="similarity"` instead of `"mmr"`.
  MMR deliberately trades relevance for diversity, which was demoting the
  actually-best-matching chunk in favor of a "different" one.
- BM25 params tuned to `k1=1.8, b=0.85` for dense clinical prose (defaults
  were the generic `k1=1.2, b=0.75`).
- The unused `EnsembleRetriever` from the original is dropped — fusion is
  handled explicitly and tunably in the retrieval function below (FIX #2).


In [7]:
langchain_docs = [
    Document(
        page_content=chunk["text"],
        metadata={
            "source": chunk["source"],
            "section": chunk["section"],
            "section_start_page": chunk["section_start_page"],
            "chunk_id": chunk["chunk_id"],
        }
    )
    for chunk in rag_chunks
]
print(f"Loaded {len(langchain_docs)} deduplicated chunks into LangChain Documents.")

print("Loading embedding model...")
embeddings = HuggingFaceEmbeddings(model_name="NeuML/pubmedbert-base-embeddings")

print("Creating vector database...")
vector_store = Chroma.from_documents(
    documents=langchain_docs,
    embedding=embeddings,
    collection_name="clinical_guidelines_v2",
    collection_metadata={"hnsw:space": "cosine"}   # force cosine distance, bounded [0, 2]
)

def get_similarity_scores(query, k, filter_=None):
    """Cosine distance -> bounded similarity, computed ourselves instead of
    trusting Chroma's default relevance_score_fn (which produced unbounded,
    wildly out-of-range values for this embedding model)."""
    results = vector_store.similarity_search_with_score(query, k=k, filter=filter_)
    return [(doc, 1 - (dist / 2)) for doc, dist in results]  # cosine distance in [0,2] -> similarity in [0,1]

print("Initializing BM25 retriever (tuned k1/b)...")
bm25_retriever = BM25Retriever.from_documents(
    langchain_docs,
    bm25_params={"k1": 1.8, "b": 0.85}   # was 1.2 / 0.75
)
bm25_retriever.k = 20

print("Initializing similarity-based vector retriever (was MMR)...")
vector_retriever = vector_store.as_retriever(
    search_type="similarity",            # was "mmr"
    search_kwargs={"k": 20}
)

print("Loading cross-encoder reranker...")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

print("Pipeline ready.")


Loaded 932 deduplicated chunks into LangChain Documents.
Loading embedding model...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/6.33k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Pipeline ready.


## 5. Retrieval — FIX #3, #4, #7, #8

- `expand_for_bm25` bridges abbreviations (`IE` → `IE infective endocarditis`)
  for the BM25 leg only — the embeddings already understand this semantically,
  so expanding the vector-search query too would just add noise.
- `detect_source_filter` restricts the vector search to one PDF when the
  query names a specific guideline ("...2023 ESC guidelines...").
- `compute_confidence` now takes **real similarity scores** from Chroma
  instead of rank position.
- `retrieve_with_rerank` adds a cross-encoder pass on top of the fused
  candidate pool as the final ranking stage.


In [8]:
import math
import re

ABBREVIATIONS = {
    "IE": "infective endocarditis",
    "ESC": "European Society of Cardiology",
    "NICE": "National Institute for Health and Care Excellence",
    "TOE": "transoesophageal echocardiography",
    "TTE": "transthoracic echocardiography",
    "GDG": "Guideline Development Group",
}

def expand_for_bm25(query: str) -> str:
    expanded = query
    for abbr, full in ABBREVIATIONS.items():
        expanded = re.sub(rf'\b{abbr}\b', f'{abbr} {full}', expanded)
    return expanded


def detect_source_filter(query: str):
    q = query.lower()
    if "esc" in q or "2023" in q:
        return {"source": "ESC.pdf"}
    if "nice" in q:
        return {"source": "NICE.pdf"}
    return None


def get_significant_words(text, min_len=4):
    return set(w.lower().strip('.,;:()') for w in text.split() if len(w) >= min_len)


def compute_confidence(query, chunk_text, bm25_rank, vector_similarity,
                        in_bm25, in_vector, bm25_total=20):
    bm25_score = (bm25_total - bm25_rank) / bm25_total if in_bm25 else 0.0
    vector_score = vector_similarity if in_vector else 0.0   # real cosine similarity now

    agreement_bonus = 0.15 if (in_bm25 and in_vector) else 0.0

    query_words = get_significant_words(query)
    chunk_words = get_significant_words(chunk_text)
    coverage = len(query_words & chunk_words) / max(len(query_words), 1)

    raw_score = (0.40 * bm25_score) + (0.30 * vector_score) + agreement_bonus + (0.15 * coverage)
    return round(min(raw_score, 1.0), 3)


def retrieve_fused(query, k=10, pool_k=20, use_source_filter=True, use_bm25_expansion=True):
    bm25_query = expand_for_bm25(query) if use_bm25_expansion else query
    bm25_results = bm25_retriever.invoke(bm25_query)

    filter_ = detect_source_filter(query) if use_source_filter else None
    vector_results_scored = get_similarity_scores(query, k=pool_k, filter_=filter_)

    bm25_ids = {doc.metadata["chunk_id"]: i for i, doc in enumerate(bm25_results)}
    vector_scores = {doc.metadata["chunk_id"]: score for doc, score in vector_results_scored}

    all_docs = {doc.metadata["chunk_id"]: doc for doc in bm25_results}
    all_docs.update({doc.metadata["chunk_id"]: doc for doc, _ in vector_results_scored})

    scored = []
    for chunk_id, doc in all_docs.items():
        in_bm25 = chunk_id in bm25_ids
        in_vector = chunk_id in vector_scores
        conf = compute_confidence(
            query, doc.page_content,
            bm25_rank=bm25_ids.get(chunk_id, 20),
            vector_similarity=vector_scores.get(chunk_id, 0.0),
            in_bm25=in_bm25, in_vector=in_vector
        )
        scored.append({"doc": doc, "confidence": conf})

    # Deterministic Tie-Breaking (sort by confidence, then chunk_id)
    scored.sort(key=lambda x: (x["confidence"], x["doc"].metadata.get("chunk_id", "")), reverse=True)
    return scored[:pool_k]


def retrieve_with_rerank(query, k=10, pool_k=20, use_source_filter=True, use_bm25_expansion=True):
    """Fusion pool -> cross-encoder rerank with Sigmoid -> top-k."""
    candidates = retrieve_fused(query, k=pool_k, pool_k=pool_k,
                                 use_source_filter=use_source_filter,
                                 use_bm25_expansion=use_bm25_expansion)
    if not candidates:
        return []

    pairs = [[query, c["doc"].page_content] for c in candidates]
    rerank_scores = reranker.predict(pairs)

    # Sigmoid calibration: maps unbounded logits stably to (0, 1) independently of candidate pool
    for c, s in zip(candidates, rerank_scores):
        s = float(s)
        c["rerank_score"] = round(1.0 / (1.0 + math.exp(-s)), 4)

    # Deterministic Tie-Breaking (sort by rerank_score, then chunk_id)
    candidates.sort(key=lambda x: (x["rerank_score"], x["doc"].metadata.get("chunk_id", "")), reverse=True)
    return candidates[:k]


## 6. Eval — original config vs improved config, side by side

Reuses the same `is_relevant` (word-overlap) check and eval set as the
original notebook so the numbers are directly comparable. Runs:

- **Baseline**: original MMR vector retriever + rank-based confidence, no
  reranking, no expansion, no source filter (mirrors `Version12PM.ipynb`
  exactly)
- **Improved**: everything above

If you don't have `evaluation_data` from the original notebook already
loaded, paste it into the cell below (same 20 Q/A pairs).


In [9]:
evaluation_data=[
  {
    "id": 1,
    "question": "What cardiac conditions does NICE consider to put a person at increased risk of infective endocarditis?",
    "answer": "Acquired valvular heart disease with stenosis or regurgitation; valve replacement; structural congenital heart disease (with specified exclusions); previous infective endocarditis; and hypertrophic cardiomyopathy.",
    "reference": "NICE Clinical Guideline 64, Recommendation 1.1.1",
    "citation": "NICE (1).pdf, lines 175-190"
  },
  {
    "id": 2,
    "question": "Is antibiotic prophylaxis recommended for patients undergoing dental procedures to prevent infective endocarditis?",
    "answer": "No. NICE recommends that antibiotic prophylaxis against infective endocarditis should not be given solely for dental procedures.",
    "reference": "NICE Clinical Guideline 64, Recommendation 1.1.3",
    "citation": "NICE (1).pdf, lines 35-44"
  },
  {
    "id": 3,
    "question": "Which non-dental procedure sites are specifically covered by NICE's recommendation against antibiotic prophylaxis?",
    "answer": "The upper and lower gastrointestinal tract; the genitourinary tract, including urological, gynaecological and obstetric procedures and childbirth; and the upper and lower respiratory tract, including ear, nose and throat procedures and bronchoscopy.",
    "reference": "NICE Clinical Guideline 64, Recommendation 1.1.3",
    "citation": "NICE (1).pdf, lines 313-324"
  },
  {
    "id": 4,
    "question": "Should chlorhexidine mouthwash be offered to prevent infective endocarditis before a dental procedure?",
    "answer": "No. NICE specifically recommends that chlorhexidine mouthwash should not be offered as prophylaxis against infective endocarditis.",
    "reference": "NICE Clinical Guideline 64, Recommendation 1.1.4",
    "citation": "NICE (1).pdf, lines 325-327"
  },
  {
    "id": 5,
    "question": "What should healthcare professionals tell patients who are at risk of infective endocarditis about prevention?",
    "answer": "They should explain the benefits and risks of antibiotic prophylaxis and why it is no longer routinely recommended, emphasize the importance of good oral health, explain symptoms that may indicate infective endocarditis and when to seek expert advice, and discuss the risks of invasive procedures including body piercing and tattooing.",
    "reference": "NICE Clinical Guideline 64, Recommendation 1.1.2",
    "citation": "NICE (1).pdf, lines 25-34"
  },
  {
    "id": 6,
    "question": "Why did NICE move away from routine antibiotic prophylaxis for infective endocarditis?",
    "answer": "The guideline found no consistent association between interventional procedures and infective endocarditis, regular toothbrushing may cause more repeated exposure to oral bacteraemia than a single dental procedure, the clinical effectiveness of prophylaxis was unproven, and prophylaxis could cause fatal anaphylaxis and was not cost-effective for dental procedures.",
    "reference": "NICE Clinical Guideline 64, Overview",
    "citation": "NICE (1).pdf, lines 381-392"
  },
  {
    "id": 7,
    "question": "What is the approximate annual incidence of infective endocarditis in the normal population according to the guideline?",
    "answer": "Fewer than 10 cases per 100,000 people per year.",
    "reference": "NICE Clinical Guideline 64, Overview",
    "citation": "NICE (1).pdf, lines 344-349"
  },
  {
    "id": 8,
    "question": "Approximately what mortality does NICE report for infective endocarditis?",
    "answer": "Approximately 20% mortality.",
    "reference": "NICE Clinical Guideline 64, Overview",
    "citation": "NICE (1).pdf, lines 344-349"
  },
  {
    "id": 9,
    "question": "Which organisms are identified as important causes of infective endocarditis in the guideline?",
    "answer": "Streptococci, Staphylococcus aureus, and enterococci are identified as important causative organisms.",
    "reference": "NICE Clinical Guideline 64",
    "citation": "NICE (1).pdf"
  },
  {
    "id": 10,
    "question": "Why might regular toothbrushing be considered a greater IE risk than a single dental procedure?",
    "answer": "Because toothbrushing occurs repeatedly and therefore produces repeated exposure to bacteraemia involving oral flora, whereas a dental procedure is a single exposure.",
    "reference": "NICE Clinical Guideline 64, Evidence to Recommendations",
    "citation": "NICE (1).pdf, lines 564-571"
  },
  {
    "id": 11,
    "question": "A patient has a fully repaired ventricular septal defect. Does NICE classify this condition as associated with increased infective endocarditis risk?",
    "answer": "No. A repaired ventricular septal defect is explicitly listed among the conditions not associated with increased infective endocarditis risk.",
    "reference": "NICE Clinical Guideline 64, Evidence Statement",
    "citation": "NICE (1).pdf, lines 651-660"
  },
  {
    "id": 12,
    "question": "Is an isolated atrial septal defect considered an infective endocarditis risk condition by NICE?",
    "answer": "No. An isolated atrial septal defect is explicitly excluded from the structural congenital heart conditions considered to be at increased risk.",
    "reference": "NICE Clinical Guideline 64, Recommendation 1.1.1",
    "citation": "NICE (1).pdf, lines 177-190"
  },
  {
    "id": 13,
    "question": "What should happen if a person at risk of infective endocarditis develops an infection?",
    "answer": "The infection should be investigated and treated promptly to reduce the risk of infective endocarditis developing.",
    "reference": "NICE Clinical Guideline 64, Recommendation 1.1.5",
    "citation": "NICE (1).pdf, lines 52-55"
  },
  {
    "id": 14,
    "question": "A patient at risk of infective endocarditis is undergoing a gastrointestinal procedure at a site with suspected infection and is already receiving antimicrobial therapy. What does NICE recommend?",
    "answer": "The antibiotic should cover organisms that cause infective endocarditis.",
    "reference": "NICE Clinical Guideline 64, Recommendation 1.1.6",
    "citation": "NICE (1).pdf, lines 56-60"
  },
  {
    "id": 15,
    "question": "What four broad categories of cardiac conditions are identified in NICE's evidence statement as associated with infective endocarditis risk?",
    "answer": "Acquired valvular heart disease with stenosis or regurgitation, valve replacement, structural congenital heart disease, and hypertrophic cardiomyopathy.",
    "reference": "NICE Clinical Guideline 64, Evidence Statement",
    "citation": "NICE (1).pdf, lines 651-655"
  },
  {
    "id": 16,
    "question": "What cardiac conditions does NICE explicitly identify as not associated with infective endocarditis risk?",
    "answer": "Isolated atrial septal defect, repaired ventricular septal defect, repaired patent ductus arteriosus, and closure devices that are judged to be endothelialised.",
    "reference": "NICE Clinical Guideline 64, Evidence Statement",
    "citation": "NICE (1).pdf, lines 656-660"
  },
  {
    "id": 17,
    "question": "What does the guideline say about the clinical evidence supporting antibiotic prophylaxis?",
    "answer": "The clinical effectiveness of antibiotic prophylaxis was not proven. The evidence base was limited, with many relevant studies being observational.",
    "reference": "NICE Clinical Guideline 64, Overview and Evidence Review",
    "citation": "NICE (1).pdf, lines 381-392"
  },
  {
    "id": 18,
    "question": "What adverse effects of antibiotic prophylaxis did the Guideline Development Group consider?",
    "answer": "They considered antibiotic-related anaphylaxis, which can be rare but potentially fatal, as well as other adverse effects and the contribution of antibiotic use to antimicrobial resistance.",
    "reference": "NICE Clinical Guideline 64, Evidence to Recommendations",
    "citation": "NICE (1).pdf, lines 544-559"
  },
  {
    "id": 19,
    "question": "What was the estimated risk of infective endocarditis after an unprotected dental procedure in adults with known predisposing cardiac conditions in the French study discussed by NICE?",
    "answer": "Approximately 1 case per 46,000 unprotected dental procedures.",
    "reference": "NICE Clinical Guideline 64, Evidence Review",
    "citation": "NICE (1).pdf, lines 430-448"
  },
  {
    "id": 20,
    "question": "Does the NICE guideline say that prophylactic antibiotics completely eliminate bacteraemia following non-dental procedures?",
    "answer": "No. The guideline states that prophylactic antibiotics can reduce, but do not eliminate, post-procedural bacteraemia.",
    "reference": "NICE Clinical Guideline 64, Evidence to Recommendations",
    "citation": "NICE (1).pdf, lines 544-553"
  }
]

In [10]:
def is_relevant(ground_truth, chunk_content):
    gt_words = get_significant_words(ground_truth)
    matches = sum(1 for w in gt_words if w in chunk_content.lower())
    return matches >= 2


# --- Baseline retriever, rebuilt here for a fair side-by-side comparison ---
baseline_vector_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 20, "fetch_k": 40, "lambda_mult": 0.5}
)
baseline_bm25_retriever = BM25Retriever.from_documents(
    langchain_docs, bm25_params={"k1": 1.2, "b": 0.75}
)
baseline_bm25_retriever.k = 20

def compute_confidence_baseline(query, chunk_text, bm25_rank, vector_rank,
                                 in_bm25, in_vector, bm25_total=20, vector_total=20):
    bm25_score = (bm25_total - bm25_rank) / bm25_total if in_bm25 else 0.0
    vector_score = (vector_total - vector_rank) / vector_total if in_vector else 0.0
    agreement_bonus = 0.15 if (in_bm25 and in_vector) else 0.0
    query_words = get_significant_words(query)
    chunk_words = get_significant_words(chunk_text)
    coverage = len(query_words & chunk_words) / max(len(query_words), 1)
    raw_score = (0.35 * bm25_score) + (0.35 * vector_score) + agreement_bonus + (0.15 * coverage)
    return round(min(raw_score, 1.0), 3)

def retrieve_baseline(query, k=10):
    bm25_results = baseline_bm25_retriever.invoke(query)
    vector_results = baseline_vector_retriever.invoke(query)
    bm25_ids = {doc.metadata["chunk_id"]: i for i, doc in enumerate(bm25_results)}
    vector_ids = {doc.metadata["chunk_id"]: i for i, doc in enumerate(vector_results)}
    all_docs = {doc.metadata["chunk_id"]: doc for doc in (bm25_results + vector_results)}
    scored = []
    for chunk_id, doc in all_docs.items():
        in_bm25 = chunk_id in bm25_ids
        in_vector = chunk_id in vector_ids
        conf = compute_confidence_baseline(
            query, doc.page_content,
            bm25_rank=bm25_ids.get(chunk_id, 20), vector_rank=vector_ids.get(chunk_id, 20),
            in_bm25=in_bm25, in_vector=in_vector
        )
        scored.append({"doc": doc, "confidence": conf})

    # Deterministic Tie-Breaking
    scored.sort(key=lambda x: (x["confidence"], x["doc"].metadata.get("chunk_id", "")), reverse=True)
    return scored[:k]


def run_eval(retrieve_fn, label, k_values=(3, 5, 10)):
    all_results = []
    for item in evaluation_data:
        query = item.get("question") or item.get("query")
        ground_truth = item["answer"]
        results = retrieve_fn(query, k=10)
        for k in k_values:
            top_k = results[:k]
            relevant_count = sum(1 for r in top_k if is_relevant(ground_truth, r["doc"].page_content))
            precision_at_k = relevant_count / k if k else 0.0
            def score_for_display(r):
                return r.get("rerank_score", r.get("confidence", 0.0))

            avg_confidence = sum(score_for_display(r) for r in top_k) / k if top_k else 0.0
            all_results.append({"K": k, "precision": precision_at_k, "avg_confidence": avg_confidence})

    print(f"\n{'='*60}\n{label}\n{'='*60}")
    summary = {}
    for k in k_values:
        scores = [r for r in all_results if r["K"] == k]
        avg_p = sum(s["precision"] for s in scores) / len(scores)
        avg_c = sum(s["avg_confidence"] for s in scores) / len(scores)
        print(f"K={k}: Avg Precision={avg_p:.3f}, Avg Confidence={avg_c:.3f}")
        summary[k] = {"precision": avg_p, "confidence": avg_c}
    return summary


baseline_summary = run_eval(retrieve_baseline, "BASELINE (original Version12PM config)")
improved_summary = run_eval(retrieve_with_rerank, "IMPROVED (this notebook)")

print(f"\n{'='*60}\nDELTA (Improved - Baseline)\n{'='*60}")
for k in (3, 5, 10):
    dp = improved_summary[k]["precision"] - baseline_summary[k]["precision"]
    print(f"K={k}: Precision {'+' if dp >= 0 else ''}{dp:.3f}")



BASELINE (original Version12PM config)
K=3: Avg Precision=0.817, Avg Confidence=0.772
K=5: Avg Precision=0.810, Avg Confidence=0.682
K=10: Avg Precision=0.775, Avg Confidence=0.534

IMPROVED (this notebook)
K=3: Avg Precision=0.900, Avg Confidence=0.987
K=5: Avg Precision=0.850, Avg Confidence=0.967
K=10: Avg Precision=0.770, Avg Confidence=0.885

DELTA (Improved - Baseline)
K=3: Precision +0.083
K=5: Precision +0.040
K=10: Precision -0.005


In [11]:
# ============================================================
# SECTION 6 — DETAILED RETRIEVAL EVALUATION
# Baseline vs Improved
# ============================================================

def is_relevant(ground_truth, chunk_content):
    gt_words = get_significant_words(ground_truth)
    matches = sum(
        1 for w in gt_words
        if w in chunk_content.lower()
    )
    return matches >= 2


# ============================================================
# BASELINE RETRIEVER
# ============================================================

baseline_vector_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 20,
        "fetch_k": 40,
        "lambda_mult": 0.5
    }
)

baseline_bm25_retriever = BM25Retriever.from_documents(
    langchain_docs,
    bm25_params={
        "k1": 1.2,
        "b": 0.75
    }
)

baseline_bm25_retriever.k = 20


def compute_confidence_baseline(
    query,
    chunk_text,
    bm25_rank,
    vector_rank,
    in_bm25,
    in_vector,
    bm25_total=20,
    vector_total=20
):

    bm25_score = (
        (bm25_total - bm25_rank) / bm25_total
        if in_bm25
        else 0.0
    )

    vector_score = (
        (vector_total - vector_rank) / vector_total
        if in_vector
        else 0.0
    )

    agreement_bonus = (
        0.15
        if (in_bm25 and in_vector)
        else 0.0
    )

    query_words = get_significant_words(query)
    chunk_words = get_significant_words(chunk_text)

    coverage = (
        len(query_words & chunk_words)
        / max(len(query_words), 1)
    )

    raw_score = (
        (0.35 * bm25_score)
        + (0.35 * vector_score)
        + agreement_bonus
        + (0.15 * coverage)
    )

    return round(
        min(raw_score, 1.0),
        3
    )


def retrieve_baseline(query, k=10):

    bm25_results = baseline_bm25_retriever.invoke(query)

    vector_results = baseline_vector_retriever.invoke(query)

    bm25_ids = {
        doc.metadata["chunk_id"]: i
        for i, doc in enumerate(bm25_results)
    }

    vector_ids = {
        doc.metadata["chunk_id"]: i
        for i, doc in enumerate(vector_results)
    }

    all_docs = {
        doc.metadata["chunk_id"]: doc
        for doc in (bm25_results + vector_results)
    }

    scored = []

    for chunk_id, doc in all_docs.items():

        in_bm25 = chunk_id in bm25_ids
        in_vector = chunk_id in vector_ids

        conf = compute_confidence_baseline(
            query=query,
            chunk_text=doc.page_content,
            bm25_rank=bm25_ids.get(chunk_id, 20),
            vector_rank=vector_ids.get(chunk_id, 20),
            in_bm25=in_bm25,
            in_vector=in_vector
        )

        scored.append({
            "doc": doc,
            "confidence": conf
        })

    # Deterministic tie-breaking
    scored.sort(
        key=lambda x: (
            x["confidence"],
            x["doc"].metadata.get("chunk_id", "")
        ),
        reverse=True
    )

    return scored[:k]


# ============================================================
# DETAILED EVALUATION FUNCTION
# ============================================================

def run_detailed_eval(
    retrieve_fn,
    label,
    k_values=(3, 5, 10)
):

    all_results = []

    print("\n" + "=" * 110)
    print(label)
    print("=" * 110)

    # --------------------------------------------------------
    # Run every evaluation question
    # --------------------------------------------------------

    for i, item in enumerate(evaluation_data, 1):

        query = (
            item.get("question")
            or item.get("query")
        )

        ground_truth = item["answer"]

        print("\n" + "=" * 110)
        print(f"Q{i}: {query}")
        print(f"Ground Truth: {ground_truth}")

        # Retrieve top 10 once
        results = retrieve_fn(
            query,
            k=10
        )

        # ----------------------------------------------------
        # Evaluate K = 3, 5, 10
        # ----------------------------------------------------

        for k in k_values:

            top_k = results[:k]

            relevant_count = sum(
                1
                for r in top_k
                if is_relevant(
                    ground_truth,
                    r["doc"].page_content
                )
            )

            precision_at_k = (
                relevant_count / k
                if k
                else 0.0
            )

            # ------------------------------------------------
            # Score used for the average
            #
            # Improved:
            #     rerank_score
            #
            # Baseline:
            #     confidence
            # ------------------------------------------------

            if top_k:

                avg_score = sum(
                    r.get(
                        "rerank_score",
                        r.get("confidence", 0.0)
                    )
                    for r in top_k
                ) / len(top_k)

            else:

                avg_score = 0.0

            all_results.append({
                "K": k,
                "precision": precision_at_k,
                "avg_score": avg_score
            })

            print(
                f"\n{'-' * 35} "
                f"Top-{k} | "
                f"Precision@{k}: {precision_at_k:.2f} | "
                f"Avg Score: {avg_score:.3f} "
                f"{'-' * 35}"
            )

            # ------------------------------------------------
            # Print every retrieved chunk
            # ------------------------------------------------

            for rank, r in enumerate(
                top_k,
                1
            ):

                meta = r["doc"].metadata

                confidence = r.get(
                    "confidence",
                    0.0
                )

                rerank_score = r.get(
                    "rerank_score",
                    None
                )

                relevant = is_relevant(
                    ground_truth,
                    r["doc"].page_content
                )

                print(
                    f"\n  [{rank}] chunk_id       : "
                    f"{meta.get('chunk_id')}"
                )

                print(
                    f"      pdf source      : "
                    f"{meta.get('source')}"
                )

                print(
                    f"      section         : "
                    f"{meta.get('section')}"
                )

                print(
                    f"      start_page      : "
                    f"{meta.get('section_start_page')}"
                )

                print(
                    f"      confidence      : "
                    f"{confidence:.3f}"
                )

                # Only the improved retriever has rerank scores
                if rerank_score is not None:

                    print(
                        f"      rerank score    : "
                        f"{rerank_score:.4f}"
                    )

                print(
                    f"      relevant        : "
                    f"{relevant}"
                )

                print(
                    f"      text            : "
                    f"{r['doc'].page_content[:400]}..."
                )

    # ========================================================
    # FINAL SUMMARY
    # ========================================================

    print("\n" + "=" * 60)
    print(f"FINAL SUMMARY — {label}")
    print("=" * 60)

    summary = {}

    for k in k_values:

        scores = [
            r
            for r in all_results
            if r["K"] == k
        ]

        avg_p = (
            sum(
                s["precision"]
                for s in scores
            )
            / len(scores)
        )

        avg_score = (
            sum(
                s["avg_score"]
                for s in scores
            )
            / len(scores)
        )

        print(
            f"K={k}: "
            f"Avg Precision={avg_p:.3f}, "
            f"Avg Score={avg_score:.3f}"
        )

        summary[k] = {
            "precision": avg_p,
            "score": avg_score
        }

    return summary, all_results


# ============================================================
# RUN BASELINE
# ============================================================

baseline_summary, baseline_results = run_detailed_eval(
    retrieve_baseline,
    "BASELINE (original Version12PM config)"
)


# ============================================================
# RUN IMPROVED RETRIEVER
# ============================================================

improved_summary, improved_results = run_detailed_eval(
    retrieve_with_rerank,
    "IMPROVED (similarity + hybrid fusion + reranking)"
)


# ============================================================
# DELTA — IMPROVED VS BASELINE
# ============================================================

print("\n" + "=" * 60)
print("DELTA (Improved - Baseline)")
print("=" * 60)

for k in (3, 5, 10):

    precision_delta = (
        improved_summary[k]["precision"]
        - baseline_summary[k]["precision"]
    )

    print(
        f"K={k}: "
        f"Precision "
        f"{'+' if precision_delta >= 0 else ''}"
        f"{precision_delta:.3f}"
    )


# ============================================================
# OPTIONAL: SCORE DELTA
# ============================================================

print("\n" + "=" * 60)
print("SCORE DELTA (Improved - Baseline)")
print("=" * 60)

for k in (3, 5, 10):

    score_delta = (
        improved_summary[k]["score"]
        - baseline_summary[k]["score"]
    )

    print(
        f"K={k}: "
        f"Score "
        f"{'+' if score_delta >= 0 else ''}"
        f"{score_delta:.3f}"
    )

Streaming output truncated to the last 5000 lines.
      relevant        : True
      text            : NICE clinical guideline 64 – Prophylaxis against infective endocarditis 30 Evidence statements The following cardiac conditions are associated with a risk of developing IE: acquired valvular heart disease with stenosis or regurgitation, valve replacement, structural congenital heart disease (including surgically corrected or palliated structural conditions) and hypertrophic cardiomyopathy. The fol...

  [2] chunk_id       : NICE.pdf_Sec_86fbb6af_Chunk_2
      pdf source      : NICE.pdf
      section         : Preexisting cardiac conditions in adults and children and their effect on the risk of developing infective endocarditis
      start_page      : 16
      confidence      : 0.877
      relevant        : True
      text            : . 2.1.3 Preexisting cardiac conditions in adults and children and their effect on the risk of developing infective endocarditis Recommendation number 1

In [32]:
!pip install -q openai pydantic

import os, json
from typing import List, Literal, Optional
from pydantic import BaseModel, Field
from openai import OpenAI
from difflib import SequenceMatcher

# ---- Configure Azure OpenAI ----
# Set these as env vars / Colab secrets -- don't hardcode the key here.
AZURE_API_KEY  = os.environ.get("OPENAI_API_KEY","")
AZURE_BASE_URL = os.environ.get("OPENAI_BASE_URL", "https://ah30309142502238-8748-resource.openai.azure.com/openai/v1")
AZURE_MODEL    = os.environ.get("OPENAI_MODEL", "o4-mini")

if not AZURE_API_KEY:
    raise ValueError("Set OPENAI_API_KEY as an environment variable or Colab secret before running this cell.")

client = OpenAI(api_key=AZURE_API_KEY, base_url=AZURE_BASE_URL)

# Kept as GEMINI_MODEL so every downstream cell (evaluate_retrieval, generate_grounded_answer)
# keeps working without further edits -- it's just the model id now.
GEMINI_MODEL = AZURE_MODEL

# ---- Startup sanity check --------------------------------------------------
# Fails loudly right now instead of producing silent "insufficient evidence"
# refusals later if the key/base_url/model id is wrong.
try:
    _ping = client.chat.completions.create(
        model=GEMINI_MODEL,
        messages=[{"role": "user", "content": "Reply with exactly: OK"}],
    )
    print(f"Azure OpenAI client OK — model '{GEMINI_MODEL}' responded: {_ping.choices[0].message.content!r}")
except Exception as e:
    raise RuntimeError(
        f"Azure OpenAI client/model check failed for model='{GEMINI_MODEL}' at base_url='{AZURE_BASE_URL}': {e}\n"
        "Fix OPENAI_API_KEY / OPENAI_BASE_URL / OPENAI_MODEL before continuing."
    ) from e

# ---- Structured output schemas ----
class RetrievalEvaluation(BaseModel):
    """Agentic evaluator output — decides scope + evidence sufficiency."""
    in_scope: bool = Field(description="True only if the question is about infective endocarditis (risk, prevention, diagnosis, management) per NICE/ESC/AHA guidance.")
    sufficient_evidence: bool = Field(description="True only if retrieved chunks specifically support answering this question.")
    relevant_chunk_ids: List[str] = Field(description="chunk_ids that are actually usable evidence for this question.")
    evidence_gap: str = Field(description="What is missing, if anything. Empty string if evidence is sufficient.")
    reasoning: str = Field(description="Brief internal justification.")

class Citation(BaseModel):
    document: str
    section: str
    page: int
    chunk_id: str
    retrieval_score: float
    excerpt: str = Field(description="Short exact excerpt (<40 words) copied from the chunk that supports the claim.")

class EvidenceItem(BaseModel):
    claim: str
    chunk_id: str = Field(description="chunk_id this specific claim is grounded in.")

class GroundedAnswer(BaseModel):
    recommendation: str = Field(description="Short, direct answer. No patient-specific treatment.")
    supporting_evidence: List[EvidenceItem]
    citations: List[Citation]
    confidence: Literal["High", "Medium", "Low", "Insufficient Evidence"]
    safety_disclaimer: str = Field(description="Notes this supports clinicians and does not replace medical judgment.")

Azure OpenAI client OK — model 'o4-mini' responded: 'OK'


In [33]:
EVALUATOR_SYSTEM_PROMPT = """You are a retrieval quality-control agent for a clinical decision-support RAG \
system that is SCOPED ONLY to Infective Endocarditis (IE) — its risk factors, prevention, diagnosis, and \
management — as covered by NICE, ESC guidelines.

Your job is NOT to answer the question. Your job is to judge the retrieval, using only the provided chunks:

1. SCOPE CHECK: Is this question about infective endocarditis? If it is about an unrelated condition, a \
different guideline topic, or asks for patient-specific diagnosis/treatment/dosage, mark in_scope = false.
2. SUFFICIENCY CHECK: Do the retrieved chunks actually and specifically support answering this question? \
Do not rely on your own medical knowledge — judge only whether the provided text supports an answer.
3. Identify exactly which chunk_ids are genuinely relevant (not just topically nearby).

Do not fix bad retrieval by inferring answers yourself. Be strict — a fluent but unsupported answer is worse \
than a refusal."""

def evaluate_retrieval(query: str, retrieved: list) -> RetrievalEvaluation:
    """
    Raises on failure instead of silently returning a fabricated 'insufficient evidence'
    result -- callers (clinical_rag_answer) catch this and tag it as an infra error,
    distinct from a genuine evaluator judgment.
    """
    context_block = "\n\n".join(
        f"[chunk_id: {r['doc'].metadata['chunk_id']}] "
        f"(source: {r['doc'].metadata['source']}, section: {r['doc'].metadata['section']}, "
        f"score: {r.get('rerank_score', r['confidence']):.3f})\n{r['doc'].page_content}"
        for r in retrieved
    )
    user_prompt = f"User question:\n{query}\n\nRetrieved chunks:\n{context_block}"

    # NOTE: o4-mini is a reasoning model and rejects a custom `temperature` -- it's
    # fixed internally, so it's simply omitted here (unlike the old Gemini call).
    response = client.beta.chat.completions.parse(
        model=GEMINI_MODEL,
        messages=[
            {"role": "system", "content": EVALUATOR_SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        response_format=RetrievalEvaluation,
    )

    choice = response.choices[0]
    if choice.message.parsed is None:
        raise ValueError(
            f"Evaluator returned no parsed structured output. "
            f"refusal={getattr(choice.message, 'refusal', None)!r} raw={choice.message.content!r}"
        )
    return choice.message.parsed

In [34]:
GENERATION_SYSTEM_PROMPT = """You are an evidence-grounded clinical decision support assistant, scoped ONLY \
to Infective Endocarditis (IE) guidance from NICE, ESC.

Rules (do not violate any of these):
- Use ONLY the retrieved guideline context provided below. Do not use outside/training knowledge.
- If the context does not fully support a claim, do not state it.
- Do not provide patient-specific diagnosis or treatment.
- Do not infer patient-specific treatment, and do not invent missing thresholds or numbers not present in context.
- Every claim in supporting_evidence must cite the exact chunk_id it came from.
- Every citation must include document, section, page, chunk_id, retrieval_score, and a short exact excerpt \
copied from that chunk — do not cite a chunk that does not actually support the claim.
- Assign confidence (High / Medium / Low / Insufficient Evidence) based on how directly and completely the \
retrieved evidence answers the question — not on your own fluency or certainty.
- Always include a safety_disclaimer stating this supports clinicians and does not replace medical judgment."""

def generate_grounded_answer(query: str, relevant_chunks: list) -> GroundedAnswer:
    context_block = "\n\n".join(
        f"[chunk_id: {r['doc'].metadata['chunk_id']}] "
        f"document: {r['doc'].metadata['source']} | section: {r['doc'].metadata['section']} | "
        f"page: {r['doc'].metadata['section_start_page']} | retrieval_score: {r.get('rerank_score', r['confidence']):.3f}\n"
        f"{r['doc'].page_content}"
        for r in relevant_chunks
    )
    user_prompt = f"Clinical question:\n{query}\n\nRetrieved evidence:\n{context_block}"

    response = client.beta.chat.completions.parse(
        model=GEMINI_MODEL,
        messages=[
            {"role": "system", "content": GENERATION_SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        response_format=GroundedAnswer,
    )

    choice = response.choices[0]
    if choice.message.parsed is None:
        raise ValueError(
            f"Generation returned no parsed structured output. "
            f"refusal={getattr(choice.message, 'refusal', None)!r} raw={choice.message.content!r}"
        )
    return choice.message.parsed

In [38]:
def excerpt_supported(excerpt: str, chunk_text: str, min_coverage: float = 0.7) -> bool:
    """
    FIX: SequenceMatcher.ratio() divides by the COMBINED length of both
    strings, so a short excerpt vs. a long chunk can never clear a 0.5
    threshold even when the excerpt is a perfect verbatim substring
    (measured ~0.33 for a real verbatim excerpt against a ~900-char chunk).
    Instead, measure what fraction of the EXCERPT is covered by the longest
    contiguous span it shares with the chunk -- independent of chunk length,
    and robust to whitespace differences and minor LLM rewording.
    """
    excerpt_norm = ' '.join(excerpt.lower().split())
    chunk_norm = ' '.join(chunk_text.lower().split())
    if not excerpt_norm:
        return False
    if excerpt_norm in chunk_norm:
        return True
    match = SequenceMatcher(None, excerpt_norm, chunk_norm).find_longest_match(
        0, len(excerpt_norm), 0, len(chunk_norm)
    )
    coverage = match.size / len(excerpt_norm)
    return coverage >= min_coverage


def _norm(s) -> str:
    return " ".join(str(s).lower().split())


def verify_and_repair_answer(answer: GroundedAnswer, relevant_chunks: list, excerpt_match_threshold=0.7):
    """
    Never trust the LLM's own citations blindly. For every citation:
      1. chunk_id must exist in what was actually retrieved (no hallucinated chunk_ids)
      2. document/section metadata is CROSS-CHECKED against that chunk's real metadata --
         but a mismatch here (case, whitespace, a retyped colon) is a copy error, not
         proof of fabrication, since chunk_id already anchors the citation to a real,
         retrieved chunk. So we REPAIR (overwrite with verified values) instead of
         dropping a valid citation over a formatting difference.
      3. the claimed excerpt must genuinely appear in the chunk (fuzzy match) -- THIS
         is the real fabrication check, and still drops the citation if it fails.
    Only genuine drops (hallucinated chunk_id, unsupported excerpt) downgrade confidence.
    """
    chunk_lookup = {r["doc"].metadata["chunk_id"]: r["doc"] for r in relevant_chunks}

    log = []
    dropped_any = False
    valid_citations = []

    for c in answer.citations:
        chunk = chunk_lookup.get(c.chunk_id)
        if chunk is None:
            log.append(f"DROPPED citation {c.chunk_id}: chunk_id not in retrieved set (hallucinated).")
            dropped_any = True
            continue

        meta = chunk.metadata
        if _norm(meta["source"]) != _norm(c.document) or _norm(meta["section"]) != _norm(c.section):
            log.append(
                f"REPAIRED citation {c.chunk_id}: metadata didn't match verbatim "
                f"(model said document={c.document!r}, section={c.section!r}); "
                f"overwritten with verified values from the retrieved chunk."
            )
            c = c.model_copy(update={
                "document": meta["source"],
                "section": meta["section"],
                "page": meta.get("section_start_page", c.page),
            })

        excerpt_present = excerpt_supported(c.excerpt, chunk.page_content, min_coverage=excerpt_match_threshold)
        if not excerpt_present:
            log.append(f"DROPPED citation {c.chunk_id}: excerpt not found in chunk text (possible fabrication).")
            dropped_any = True
            continue

        valid_citations.append(c)

    valid_ids = {c.chunk_id for c in valid_citations}
    valid_evidence = [e for e in answer.supporting_evidence if e.chunk_id in valid_ids]
    if len(valid_evidence) < len(answer.supporting_evidence):
        n_dropped = len(answer.supporting_evidence) - len(valid_evidence)
        log.append(f"DROPPED_EVIDENCE {n_dropped} supporting_evidence item(s) with no valid citation.")
        dropped_any = True

    repaired = answer.model_copy(update={"citations": valid_citations, "supporting_evidence": valid_evidence})

    if dropped_any:
        confidence_order = ["Insufficient Evidence", "Low", "Medium", "High"]
        idx = confidence_order.index(repaired.confidence)
        repaired.confidence = confidence_order[max(0, idx - 1)]
        repaired.safety_disclaimer += " (Confidence downgraded: one or more citations failed verification.)"

    return repaired, log

In [39]:
def build_refusal(query: str, evaluation: RetrievalEvaluation, retrieved: list, reason: str, extra_detail=None) -> dict:
    if reason == "out_of_scope":
        message = ("This assistant only answers questions about infective endocarditis (risk, prevention, "
                    "diagnosis, and management) based on the NICE, ESC, and AHA guidelines it was built on. "
                    "This question falls outside that scope.")
    elif reason == "citation_verification_failed":
        message = ("The retrieved guidelines do not provide evidence that could be reliably traced and cited "
                    "for this question. Please consult the relevant clinical guideline or a qualified clinician.")
    else:  # insufficient_evidence
        message = ("The retrieved guidelines do not provide sufficient evidence to answer this question "
                    "reliably. Please consult the relevant clinical guideline or a qualified clinician.")

    found = [r["doc"].metadata["section"] for r in retrieved[:3]]

    evidence_gap = evaluation.evidence_gap
    if extra_detail:
        detail_str = "; ".join(extra_detail) if isinstance(extra_detail, list) else str(extra_detail)
        evidence_gap = f"{evidence_gap} | verification_log: {detail_str}" if evidence_gap else f"verification_log: {detail_str}"

    return {
        "query": query,
        "refused": True,
        "reason": reason,
        "message": message,
        "evidence_found_nearby": found,
        "evidence_gap": evidence_gap,
        "reasoning": evaluation.reasoning,
        "confidence": "Insufficient Evidence",
    }


def build_error_refusal(query: str, reason: str, detail: str, retrieved: list = None) -> dict:
    found = [r["doc"].metadata["section"] for r in (retrieved or [])[:3]]
    return {
        "query": query,
        "refused": True,
        "reason": reason,
        "message": (
            "This question could not be processed due to a system error, not a lack of "
            "evidence. Please retry, and if it persists, check the API configuration."
        ),
        "evidence_found_nearby": found,
        "evidence_gap": detail,
        "reasoning": "",
        "confidence": "Insufficient Evidence",
    }

In [17]:
import re

EMERGENCY_PATTERNS = [
    r"\bchest pain\b", r"\bcan'?t breathe\b", r"\bdifficulty breathing\b",
    r"\bsevere\b.*\bpain\b", r"\bemergency\b", r"\bcall (an )?ambulance\b",
    r"\bunconscious\b", r"\bpassing out\b", r"\bsevere bleeding\b",
    r"\bright now\b", r"\bhappening now\b", r"\bI think I have\b",
]

PATIENT_SPECIFIC_PATTERNS = [
    r"\bmy (patient|son|daughter|mother|father|wife|husband)\b",
    r"\b\d{1,3}[- ]year[- ]old\b",
    r"\bshould (he|she|I|my patient) (take|get|have)\b",
    r"\bwhat dose\b", r"\bhow much (should|do I)\b",
    r"\bis it safe for (him|her|me)\b",
]

def classify_input_risk(query: str) -> dict:
    q = query.lower()
    for pattern in EMERGENCY_PATTERNS:
        if re.search(pattern, q):
            return {"risk": "refuse", "reason": "emergency_pattern_detected", "matched": pattern}
    for pattern in PATIENT_SPECIFIC_PATTERNS:
        if re.search(pattern, q):
            return {"risk": "needs_caution", "reason": "patient_specific_query", "matched": pattern}
    return {"risk": "allowed", "reason": "no_risk_pattern_matched", "matched": None}


def build_emergency_refusal(query: str, classification: dict) -> dict:
    return {
        "query": query,
        "refused": True,
        "reason": "emergency_input_blocked",
        "message": (
            "This looks like it may describe an urgent or emergency situation. "
            "This assistant cannot provide emergency medical guidance. "
            "Please contact emergency services or a healthcare professional immediately."
        ),
        "evidence_found_nearby": [],
        "evidence_gap": classification["reason"],
        "confidence": "Insufficient Evidence",
    }


RETRIEVAL_CONFIDENCE_THRESHOLD = 0.30

def check_retrieval_threshold(retrieved: list, threshold: float = RETRIEVAL_CONFIDENCE_THRESHOLD) -> dict:
    if not retrieved:
        return {"passed": False, "top_score": 0.0, "reason": "no_chunks_retrieved"}
    top_score = retrieved[0].get("rerank_score", retrieved[0].get("confidence", 0.0))
    if top_score < threshold:
        return {"passed": False, "top_score": top_score, "reason": "below_confidence_threshold"}
    return {"passed": True, "top_score": top_score, "reason": "ok"}


def build_low_confidence_refusal(query: str, threshold_check: dict) -> dict:
    return {
        "query": query,
        "refused": True,
        "reason": "low_retrieval_confidence",
        "message": (
            "No sufficiently relevant guideline content was found for this question "
            f"(top retrieval score: {threshold_check['top_score']:.3f}, "
            f"minimum required: {RETRIEVAL_CONFIDENCE_THRESHOLD}). "
            "Please consult the relevant clinical guideline or a qualified clinician."
        ),
        "evidence_found_nearby": [],
        "evidence_gap": "retrieval_below_threshold",
        "confidence": "Insufficient Evidence",
    }

In [35]:
import statistics

diag_rows = []
for item in evaluation_data:
    query = item.get("question") or item.get("query")
    top = retrieve_with_rerank(query, k=1)
    if not top:
        diag_rows.append((query, None, None))
        continue
    top_doc = top[0]["doc"]
    relevant = is_relevant(item["answer"], top_doc.page_content)
    diag_rows.append((query, top[0]["rerank_score"], relevant))

scores_relevant = [s for _, s, rel in diag_rows if s is not None and rel]
scores_not_relevant = [s for _, s, rel in diag_rows if s is not None and not rel]

print(f"{'Query':<70} {'rerank_score':>12}  {'top-1 relevant?'}")
for q, s, rel in diag_rows:
    q_short = (q[:67] + "...") if len(q) > 70 else q
    print(f"{q_short:<70} {s if s is not None else float('nan'):>12.3f}  {rel}")

print()
if scores_relevant:
    print(f"Top-1 RELEVANT     rerank_score: min={min(scores_relevant):.3f}  "
          f"median={statistics.median(scores_relevant):.3f}  max={max(scores_relevant):.3f}")
if scores_not_relevant:
    print(f"Top-1 NOT relevant  rerank_score: min={min(scores_not_relevant):.3f}  "
          f"median={statistics.median(scores_not_relevant):.3f}  max={max(scores_not_relevant):.3f}")
print(f"\nCurrent RETRIEVAL_CONFIDENCE_THRESHOLD = {RETRIEVAL_CONFIDENCE_THRESHOLD}")
print("If the 'relevant' scores above cluster BELOW this threshold, that's why you're")
print("seeing low-confidence refusals despite good retrieval -- lower the threshold to")
print("just under the minimum of the 'relevant' distribution (with some margin).")

Query                                                                  rerank_score  top-1 relevant?
What cardiac conditions does NICE consider to put a person at incre...        0.999  True
Is antibiotic prophylaxis recommended for patients undergoing denta...        1.000  True
Which non-dental procedure sites are specifically covered by NICE's...        0.999  True
Should chlorhexidine mouthwash be offered to prevent infective endo...        1.000  True
What should healthcare professionals tell patients who are at risk ...        1.000  True
Why did NICE move away from routine antibiotic prophylaxis for infe...        0.969  True
What is the approximate annual incidence of infective endocarditis ...        0.994  True
Approximately what mortality does NICE report for infective endocar...        0.992  False
Which organisms are identified as important causes of infective end...        0.992  True
Why might regular toothbrushing be considered a greater IE risk tha...        1.000  Tru

In [29]:
def clinical_rag_answer(query: str, k: int = 8) -> dict:
    # 0. Deterministic input risk classification (no LLM, runs first)
    risk_check = classify_input_risk(query)
    if risk_check["risk"] == "refuse":
        return build_emergency_refusal(query, risk_check)

    # 1. Retrieval
    retrieved = retrieve_with_rerank(query, k=k)

    # 1b. Deterministic numeric confidence gate (no LLM)
    threshold_check = check_retrieval_threshold(retrieved)
    if not threshold_check["passed"]:
        return build_low_confidence_refusal(query, threshold_check)

    # 2. Agentic evaluation — scope + sufficiency
    # Evaluator failures (bad key, bad model id, network error, etc.) are caught
    # HERE and reported as "evaluator_error" -- never silently relabeled as
    # "insufficient_evidence". If this prints on every question, the problem is
    # infrastructure, not retrieval quality.
    try:
        evaluation = evaluate_retrieval(query, retrieved)
    except Exception as e:
        print(f"[evaluator_error] query={query!r} error={e}")
        return build_error_refusal(query, "evaluator_error", str(e), retrieved)

    if not evaluation.in_scope:
        return build_refusal(query, evaluation, retrieved, "out_of_scope")
    if not evaluation.sufficient_evidence or not evaluation.relevant_chunk_ids:
        return build_refusal(query, evaluation, retrieved, "insufficient_evidence")

    # 3. Restrict generation to evaluator-approved chunks only
    relevant = [r for r in retrieved if r["doc"].metadata["chunk_id"] in evaluation.relevant_chunk_ids]
    if not relevant:
        # Evaluator approved chunk_ids that match nothing actually retrieved
        # (e.g. truncated/hallucinated id) -- a different failure mode than
        # "no evidence exists", so it gets its own reason.
        return build_error_refusal(
            query, "evaluator_chunk_id_mismatch",
            f"evaluator relevant_chunk_ids={evaluation.relevant_chunk_ids!r} matched none of the "
            f"retrieved chunk_ids={[r['doc'].metadata['chunk_id'] for r in retrieved]!r}",
            retrieved,
        )

    needs_caution = risk_check["risk"] == "needs_caution"

    try:
        answer = generate_grounded_answer(query, relevant)
    except Exception as e:
        print(f"[generation_error] query={query!r} error={e}")
        return build_error_refusal(query, "generation_error", str(e), retrieved)

    answer, verification_log = verify_and_repair_answer(answer, relevant)

    if not answer.citations:
        return build_refusal(query, evaluation, retrieved, "citation_verification_failed")

    if needs_caution:
        answer.safety_disclaimer += (
            " NOTE: This question appears patient-specific. This assistant provides "
            "general guideline information only and cannot make individualized treatment decisions."
        )

    return {
        "query": query,
        "refused": False,
        "recommendation": answer.recommendation,
        "supporting_evidence": [e.model_dump() for e in answer.supporting_evidence],
        "citations": [c.model_dump() for c in answer.citations],
        "confidence": answer.confidence,
        "safety_disclaimer": answer.safety_disclaimer,
        "verification_log": verification_log,
        "input_risk": risk_check["risk"],
    }

In [30]:
def print_clinical_answer(result: dict):
    print("\n" + "=" * 100)
    print(f"Q: {result['query']}")
    print("=" * 100)
    if result["refused"]:
        print(f"\n[REFUSED — {result['reason']}]\n{result['message']}")
        if result.get("evidence_found_nearby"):
            print(f"\nNearby sections found: {result['evidence_found_nearby']}")
        if result.get("evidence_gap"):
            print(f"\nEvidence gap / error detail: {result['evidence_gap']}")
        if result.get("reasoning"):
            print(f"Evaluator reasoning: {result['reasoning']}")
        return
    print(f"\nRECOMMENDATION:\n{result['recommendation']}")
    print(f"\nSUPPORTING EVIDENCE:")
    for e in result["supporting_evidence"]:
        print(f"  - {e['claim']}  [chunk: {e['chunk_id']}]")
    print(f"\nCITATIONS:")
    for c in result["citations"]:
        print(f"  - {c['document']}, \"{c['section']}\", p.{c['page']} (score={c['retrieval_score']:.2f}, chunk={c['chunk_id']}))")
        print(f"    excerpt: \"{c['excerpt']}\"")
    print(f"\nCONFIDENCE: {result['confidence']}")
    print(f"DISCLAIMER: {result['safety_disclaimer']}")
    if result["verification_log"]:
        print(f"\n[Verifier notes: {result['verification_log']}]")

In [40]:
test_queries = [
    evaluation_data[0]["question"],   # in-scope, should have strong evidence
    evaluation_data[5]["question"],   # in-scope
    "What is the recommended chemotherapy regimen for stage 2 breast cancer?",  # out-of-scope
    "Should my 45-year-old patient with a repaired VSD get antibiotics before a root canal next week?",  # patient-specific -> should refuse or heavily caveat
]

for q in test_queries:
    result = clinical_rag_answer(q, k=8)
    print_clinical_answer(result)


Q: What cardiac conditions does NICE consider to put a person at increased risk of infective endocarditis?

RECOMMENDATION:
NICE considers the following cardiac conditions to put a person at increased risk of infective endocarditis: acquired valvular heart disease with stenosis or regurgitation; valve replacement; structural congenital heart disease (including surgically corrected or palliated conditions, but excluding isolated atrial septal defect, fully repaired ventricular septal defect or fully repaired patent ductus arteriosus, and endothelialised closure devices); previous infective endocarditis; and hypertrophic cardiomyopathy.

SUPPORTING EVIDENCE:
  - NICE considers acquired valvular heart disease with stenosis or regurgitation to increase risk of infective endocarditis.  [chunk: NICE.pdf_Sec_29061219_Chunk_1]
  - NICE considers valve replacement to increase risk of infective endocarditis.  [chunk: NICE.pdf_Sec_29061219_Chunk_1]
  - NICE considers structural congenital heart 

In [41]:
import statistics
from collections import Counter

def run_full_pipeline_evaluation(eval_data: list, k: int = 8):
    total_citations_checked = 0
    total_correct_citations = 0
    total_claims_generated = 0
    total_unsupported_claims = 0
    refusal_count = 0
    refusal_reasons = Counter()
    results_log = []

    for item in eval_data:
        query = item["question"]
        result = clinical_rag_answer(query, k=k)

        if result["refused"]:
            refusal_count += 1
            refusal_reasons[result["reason"]] += 1
            results_log.append({"query": query, "refused": True, "reason": result["reason"]})
            continue

        n_dropped = sum(1 for log in result["verification_log"] if log.startswith("DROPPED citation"))
        n_valid = len(result["citations"])
        n_total_citations = n_valid + n_dropped
        total_citations_checked += n_total_citations
        total_correct_citations += n_valid

        n_dropped_claims = sum(1 for log in result["verification_log"] if log.startswith("DROPPED_EVIDENCE"))
        n_valid_claims = len(result["supporting_evidence"])
        n_total_claims = n_valid_claims + n_dropped_claims
        total_claims_generated += n_total_claims
        total_unsupported_claims += n_dropped_claims

        results_log.append({
            "query": query, "refused": False,
            "citations_valid": n_valid, "citations_total": n_total_citations,
            "claims_valid": n_valid_claims, "claims_total": n_total_claims,
        })

    citation_accuracy = (total_correct_citations / total_citations_checked) if total_citations_checked else None
    faithfulness_rate = (total_unsupported_claims / total_claims_generated) if total_claims_generated else 0.0

    print("=" * 60)
    print("EVALUATION DASHBOARD")
    print("=" * 60)
    print(f"Total questions evaluated     : {len(eval_data)}")
    print(f"Refused                       : {refusal_count}")
    print(f"Answered                      : {len(eval_data) - refusal_count}")
    if refusal_reasons:
        print("Refusal breakdown             :")
        infra_reasons = {"evaluator_error", "generation_error", "evaluator_chunk_id_mismatch"}
        for reason, count in refusal_reasons.most_common():
            flag = "  <-- infra failure, not a real evidence gap" if reason in infra_reasons else ""
            print(f"    {reason:<30}: {count}{flag}")
    print(f"Citation Accuracy             : {citation_accuracy:.3f}" if citation_accuracy is not None else "Citation Accuracy: N/A")
    print(f"Unsupported Claim Rate        : {faithfulness_rate:.3f}")
    print("=" * 60)

    return {
        "citation_accuracy": citation_accuracy,
        "unsupported_claim_rate": faithfulness_rate,
        "refusal_count": refusal_count,
        "refusal_reasons": dict(refusal_reasons),
        "results_log": results_log,
    }


dashboard_results = run_full_pipeline_evaluation(evaluation_data, k=8)

EVALUATION DASHBOARD
Total questions evaluated     : 20
Refused                       : 3
Answered                      : 17
Refusal breakdown             :
    insufficient_evidence         : 2
    citation_verification_failed  : 1
Citation Accuracy             : 0.935
Unsupported Claim Rate        : 0.059
